# Chapter 01 — Apache Beam & Google Cloud Dataflow Fundamentals

Welcome to the interactive notebook for **Chapter 01**!

> 📖 **Concept Reference**: For comprehensive theoretical deep dives, Beam architecture, and transform naming rules, see [Chapter 01 README](README.md).

### Interactive Workflow
1. Verify the Python virtual environment and Apache Beam SDK.
2. Build and execute an in-memory pipeline (`beam.Create`) with `DirectRunner`.
3. Read and transform the retail sales dataset with custom `DoFn` and `CombinePerKey`.
4. Transition from local execution to cloud deployment on **Google Cloud Dataflow**.

## 1. Quick Verification: Apache Beam SDK

Confirm that Apache Beam and its Google Cloud dependencies are active in the kernel.
*(If you encounter import errors, see the setup guide in [README.md](README.md#environment-setup--activating-the-kernel)).*

## 2. Environment Setup & Activating the Kernel with `uv`

This project uses **[`uv`](https://docs.astral.sh/uv/)** to manage dependencies in an isolated `.venv` virtual environment.

### How to Activate the Environment

#### Option A: In the Terminal
If you are running scripts from the command line, activate the environment with:
```bash
# macOS / Linux:
source .venv/bin/activate

# Or execute any command directly without manual activation:
uv run python pipelines/sales_pipeline.py --help
```

#### Option B: In This Jupyter Notebook (Kernel Selection)
If you run a cell and see `ModuleNotFoundError: No module named 'apache_beam'`, the notebook is attached to your global system Python instead of the `.venv`.

1. **Register the kernel** (run this once from your terminal in the repository root):
   ```bash
   uv run python -m ipykernel install --user --name dataflow-workshops --display-name "Python (dataflow-workshops)"
   ```
2. **Select the kernel in your IDE**:
   * Look at the top-right corner of this notebook editor window.
   * Click the **Kernel Picker** button.
   * Select **`Python (dataflow-workshops)`** or browse to **`.venv/bin/python`**.

--- 

### Verification Cell
Run the cell below. If it executes without error and displays an interpreter path containing `.venv`, you are correctly attached to the project environment!

In [1]:
import os
import sys

# 1. Verify Python executable location
print(f"Python Executable: {sys.executable}")
if ".venv" in sys.executable:
    print("✅ Success: Notebook is attached to the project's isolated .venv environment!")
else:
    print("⚠️ Warning: Notebook is running on system Python! Please select 'Python (dataflow-workshops)' from the top-right kernel picker.")

# 2. Import and verify Apache Beam
try:
    import apache_beam as beam
    print(f"✅ Apache Beam Version: {beam.__version__}")
except ImportError as e:
    print(f"❌ Failed to import apache_beam: {e}")
    print("Run 'uv sync' in your terminal and select the registered kernel.")

Python Executable: /Users/zelda.luconi/Medium/google-products/dataflow-workshops/.venv/bin/python3
✅ Success: Notebook is attached to the project's isolated .venv environment!
✅ Apache Beam Version: 2.77.0


## 3. Your First In-Memory Beam Pipeline (`beam.Create`)

To understand how Beam constructs and executes a Directed Acyclic Graph (DAG), we will create an in-memory `PCollection` using `beam.Create()`.

### The Anatomy of Beam Pipeline Syntax

In Apache Beam, pipelines are constructed using the pipe operator (`|`) and the right-shift operator (`>>`):

```python
[Output PCollection] = [Input PCollection] | "Unique Transform Name" >> [PTransform]
```

| Component | Type | Visual Shape | Description |
| :--- | :--- | :--- | :--- |
| **`PCollection`** | **Data** | 🛢️ Cylinder / Oval `[( ... )]` | An immutable, distributed collection of elements (bounded or unbounded). |
| **`PTransform`** | **Operation** | ⚙️ Rectangle `[ ... ]` | A processing operation that reads from an input `PCollection` and writes to an output `PCollection`. |
| **Pipe (`\|`)** | **Application** | $\rightarrow$ Arrow | Applies a transformation to the preceding collection. |
| **Shift (`>>`)** | **Naming** | Label binding | Attaches a unique human-readable label to the transform node. |

--- 

### What Does the "Unique Transform Name" Stand For?

In `p | "FilterEven" >> beam.Filter(...)`, the string `"FilterEven"` is not just a comment—it is a critical runtime identifier:

1. **DAG Graph Node Label**: Apache Beam compiles your code into an abstract execution graph. This string becomes the name of that node in the graph.
2. **Google Cloud Dataflow Console UI**: When deployed to Google Cloud, Dataflow renders an interactive visual pipeline graph. Every stage box in the console is labeled with this exact name, allowing you to visually inspect throughput, execution stages, and execution duration.
3. **Cloud Monitoring & Cloud Logging Metrics**: All Dataflow metrics (wall time, CPU utilization, element counters, throughput, and custom `beam.metrics`) are scoped by this name (e.g. `dataflow.googleapis.com/job/element_count` tagged with `step="FilterEven"`).
4. **Uniqueness Constraint**: Every transform name **must be unique** within its scope. If you declare two steps with the same name, Beam throws a compilation error:
   ```text
   ValueError: A transform with name "FilterEven" already exists in this pipeline
   ```
5. **Streaming Checkpoints & Pipeline Updates**: In production streaming pipelines, Dataflow tracks in-flight window state and timers keyed by the transform's full hierarchical name. When performing zero-downtime pipeline updates (`--update`), matching names allows Dataflow to preserve state across new code deployments.

--- 

### Pipeline Execution Graph (DAG)

Notice how the **`PCollection`** (data in motion, cylinder shape) and **`PTransform`** (processing steps, rectangle box) alternate in the execution graph:

```mermaid
flowchart LR
    classDef pcoll fill:#e8f0fe,stroke:#4285f4,stroke-width:2px,color:#1a73e8,font-weight:bold;
    classDef ptrans fill:#fef7e0,stroke:#f9ab00,stroke-width:2px,color:#b06000,font-weight:bold;

    PIPE[("Pipeline Root")]:::pcoll
    T_CREATE["'CreateNumbers'<br/><i>beam.Create</i>"]:::ptrans
    PC_ALL[("Numbers 1-10")]:::pcoll
    T_FILTER["'FilterEven'<br/><i>beam.Filter</i>"]:::ptrans
    PC_EVENS[("Even Numbers")]:::pcoll
    T_MAP["'MultiplyByTen'<br/><i>beam.Map(n*10)</i>"]:::ptrans
    PC_MULT[("Multiplied 20-100")]:::pcoll
    T_PRINT["'PrintResults'<br/><i>beam.Map(print)</i>"]:::ptrans
    PC_OUT[("Stdout")]:::pcoll

    PIPE --> T_CREATE --> PC_ALL --> T_FILTER --> PC_EVENS --> T_MAP --> PC_MULT --> T_PRINT --> PC_OUT
```


Now let's execute this pipeline with Apache Beam:

In [2]:
numbers = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

with beam.Pipeline() as p:
    (
        p
        | "CreateNumbers" >> beam.Create(numbers)
        | "FilterEven" >> beam.Filter(lambda n: n % 2 == 0)
        | "MultiplyByTen" >> beam.Map(lambda n: n * 10)
        | "PrintResults" >> beam.Map(lambda n: print(f"Even * 10: {n}"))
    )

Even * 10: 20
Even * 10: 40
Even * 10: 60
Even * 10: 80
Even * 10: 100


> **Key Takeaway**: When entering the `with beam.Pipeline() as p:` context, Beam constructs the abstract DAG graph. The pipeline is evaluated and executed only when the context exits!

## 4. Reading External Data: The Retail Sales Dataset

In our workshop storyline, we process a retail transactions dataset (`data/sales_data.csv`).

Let's inspect the first few lines of the CSV file using Python.

In [3]:
from pathlib import Path

csv_path = Path("data/sales_data.csv")
print(csv_path.read_text())

transaction_id,product_id,amount,transaction_date
TID001,PROD_A,150.75,2025-06-12
TID002,PROD_B,200.00,2025-06-12
TID003,PROD_A,75.20,2025-06-11
TID004,PROD_C,300.50,2025-06-12
TID005,PROD_B,120.99,2025-06-13
TID006,PROD_A,99.99,2025-06-12
TID007,PROD_D,50.00,2025-06-11
TID008,PROD_C,450.00,2025-06-13
TID009,PROD_A,25.00,2025-06-12
TID010,PROD_B,310.25,2025-06-12



Each row contains:
* `transaction_id`: Unique identifier (e.g., `TID001`)
* `product_id`: Product SKU (e.g., `PROD_A`)
* `amount`: Revenue in USD (e.g., `150.75`)
* `transaction_date`: Date string in `YYYY-MM-DD` format

Now let's use `beam.io.ReadFromText` to read these rows into a `PCollection`.

In [4]:
with beam.Pipeline() as p:
    (
        p
        | "ReadLines" >> beam.io.ReadFromText("data/sales_data.csv")
        | "PrintRawLines" >> beam.Map(lambda line: print(f"Raw line: {line}"))
    )

Raw line: transaction_id,product_id,amount,transaction_date
Raw line: TID001,PROD_A,150.75,2025-06-12
Raw line: TID002,PROD_B,200.00,2025-06-12
Raw line: TID003,PROD_A,75.20,2025-06-11
Raw line: TID004,PROD_C,300.50,2025-06-12
Raw line: TID005,PROD_B,120.99,2025-06-13
Raw line: TID006,PROD_A,99.99,2025-06-12
Raw line: TID007,PROD_D,50.00,2025-06-11
Raw line: TID008,PROD_C,450.00,2025-06-13
Raw line: TID009,PROD_A,25.00,2025-06-12
Raw line: TID010,PROD_B,310.25,2025-06-12


## 5. Custom Element-Wise Processing: `DoFn` and `beam.ParDo`

We define custom element-wise business logic using `beam.DoFn` subclasses and execute them across distributed workers using `beam.ParDo`.
*(For the complete architectural deep dive, lifecycle phases, and comparison matrix, see [README.md](README.md#core-concepts-deep-dive-dofn-vs-beam-pardo)).*


In [5]:
class ParseSalesRecordFn(beam.DoFn):
    """Parses a CSV row into a structured dictionary, skipping headers."""
    def process(self, line: str):
        line = line.strip()
        if not line or line.startswith("transaction_id"):
            return  # Skip empty lines and CSV header
        
        parts = [p.strip() for p in line.split(",")]
        if len(parts) == 4:
            try:
                yield {
                    "transaction_id": parts[0],
                    "product_id": parts[1],
                    "amount": float(parts[2]),
                    "transaction_date": parts[3],
                }
            except ValueError:
                pass

class FilterByDateFn(beam.DoFn):
    """Filters transactions matching a target date."""
    def __init__(self, target_date: str):
        self.target_date = target_date
        
    def process(self, record: dict):
        if record.get("transaction_date") == self.target_date:
            yield record

Let's test our `DoFn` classes inside a pipeline:

In [6]:
with beam.Pipeline() as p:
    (
        p
        | "ReadCSV" >> beam.io.ReadFromText("data/sales_data.csv")
        | "ParseRecords" >> beam.ParDo(ParseSalesRecordFn())
        | "FilterDate" >> beam.ParDo(FilterByDateFn(target_date="2025-06-12"))
        | "PrintFiltered" >> beam.Map(lambda r: print(f"Filtered: {r}"))
    )

Filtered: {'transaction_id': 'TID001', 'product_id': 'PROD_A', 'amount': 150.75, 'transaction_date': '2025-06-12'}
Filtered: {'transaction_id': 'TID002', 'product_id': 'PROD_B', 'amount': 200.0, 'transaction_date': '2025-06-12'}
Filtered: {'transaction_id': 'TID004', 'product_id': 'PROD_C', 'amount': 300.5, 'transaction_date': '2025-06-12'}
Filtered: {'transaction_id': 'TID006', 'product_id': 'PROD_A', 'amount': 99.99, 'transaction_date': '2025-06-12'}
Filtered: {'transaction_id': 'TID009', 'product_id': 'PROD_A', 'amount': 25.0, 'transaction_date': '2025-06-12'}
Filtered: {'transaction_id': 'TID010', 'product_id': 'PROD_B', 'amount': 310.25, 'transaction_date': '2025-06-12'}


## 6. Aggregations: `CombinePerKey`

We aggregate revenue per product key using `beam.CombinePerKey(sum)`, performing partial combiner reductions locally on workers before distributed shuffle.
*(For a comparative breakdown of `GroupByKey` vs. `CombinePerKey`, see [README.md](README.md)).*

In [7]:
with beam.Pipeline() as p:
    (
        p
        | "ReadCSV" >> beam.io.ReadFromText("data/sales_data.csv")
        | "ParseRecords" >> beam.ParDo(ParseSalesRecordFn())
        | "FilterDate" >> beam.ParDo(FilterByDateFn(target_date="2025-06-12"))
        | "MapKeyVal" >> beam.Map(lambda r: (r["product_id"], r["amount"]))
        | "SumRevenue" >> beam.CombinePerKey(sum)
        | "PrintSummary" >> beam.Map(lambda kv: print(f"Product: {kv[0]} -> Total Revenue: ${kv[1]:.2f}"))
    )

Product: PROD_A -> Total Revenue: $275.74
Product: PROD_B -> Total Revenue: $510.25
Product: PROD_C -> Total Revenue: $300.50


## 7. Writing Results: `beam.io.WriteToText`

In production, results are written to partitioned files on **Cloud Storage (`gs://`)** or tables in **BigQuery**.

Here is the complete DAG of our end-to-end retail pipeline, showing how **`PCollection`** (data cylinders) and **`PTransform`** (operation boxes) connect:


```mermaid
flowchart LR
    classDef pcoll fill:#e8f0fe,stroke:#4285f4,stroke-width:2px,color:#1a73e8,font-weight:bold;
    classDef ptrans fill:#fef7e0,stroke:#f9ab00,stroke-width:2px,color:#b06000,font-weight:bold;

    subgraph S1["1. Ingestion"]
        RAW[("sales_data.csv")]:::pcoll
        T_READ["'ReadCSV'<br/><i>ReadFromText</i>"]:::ptrans
        PC_RAW[("Raw Lines")]:::pcoll
        RAW --> T_READ --> PC_RAW
    end

    subgraph S2["2. Parse & Filter"]
        T_PARSE["'ParseRecords'<br/><i>ParseSalesRecordFn</i>"]:::ptrans
        PC_PARSED[("Parsed Dicts")]:::pcoll
        T_FILTER["'FilterDate'<br/><i>FilterByDateFn</i>"]:::ptrans
        PC_FILTERED[("Target Date")]:::pcoll
        T_PARSE --> PC_PARSED --> T_FILTER --> PC_FILTERED
    end

    subgraph S3["3. Aggregate"]
        T_MAP["'ExtractProductAmount'<br/><i>Map(extract_kv)</i>"]:::ptrans
        PC_KV[("(product_id, amount)")]:::pcoll
        T_COMBINE["'AggregateRevenue'<br/><i>CombinePerKey(sum)</i>"]:::ptrans
        PC_AGG[("(product_id, total)")]:::pcoll
        T_MAP --> PC_KV --> T_COMBINE --> PC_AGG
    end

    subgraph S4["4. Format & Sink"]
        T_FORMAT["'FormatOutput'<br/><i>Map(format_output)</i>"]:::ptrans
        PC_STR[("Formatted Strings")]:::pcoll
        T_WRITE["'WriteResults'<br/><i>WriteToText</i>"]:::ptrans
        OUT[("revenue_summary-*.txt")]:::pcoll
        T_FORMAT --> PC_STR --> T_WRITE --> OUT
    end

    PC_RAW --> T_PARSE
    PC_FILTERED --> T_MAP
    PC_AGG --> T_FORMAT
```


Now let's execute the full pipeline:

In [8]:
output_prefix = "output/notebook_sales_summary"

with beam.Pipeline() as p:
    (
        p
        | "ReadCSV" >> beam.io.ReadFromText("data/sales_data.csv")
        | "ParseRecords" >> beam.ParDo(ParseSalesRecordFn())
        | "FilterDate" >> beam.ParDo(FilterByDateFn(target_date="2025-06-12"))
        | "MapKeyVal" >> beam.Map(lambda r: (r["product_id"], r["amount"]))
        | "SumRevenue" >> beam.CombinePerKey(sum)
        | "FormatOutput" >> beam.Map(lambda kv: f"product_id={kv[0]}, total_revenue={kv[1]:.2f}")
        | "WriteResults" >> beam.io.WriteToText(output_prefix, file_name_suffix=".txt")
    )

print("Pipeline completed successfully!")

Pipeline completed successfully!


In [9]:
# Let's inspect the generated file(s)
for file in Path("output").glob("notebook_sales_summary*.txt"):
    print(f"--- Contents of {file.name} ---")
    print(file.read_text())

--- Contents of notebook_sales_summary-00000-of-00001.txt ---
product_id=PROD_A, total_revenue=275.74
product_id=PROD_B, total_revenue=510.25
product_id=PROD_C, total_revenue=300.50



## 8. Submitting to Google Cloud Dataflow (`DataflowRunner`)

To scale out onto Google-managed worker VMs, we pass `DataflowRunner` with Cloud Storage staging paths.
*(For complete runner flags and project permissions, see [README.md](README.md#2-run-on-google-cloud-dataflow)).*

## 9. Running via Modular Scripts

We have packaged the entire pipeline into a production-ready driver script [`pipelines/sales_pipeline.py`](./pipelines/sales_pipeline.py).

You can execute it using the accompanying shell scripts:

```bash
# 1. Run locally with DirectRunner:
bash 01_beam_fundamentals/scripts/01_run_local.sh

# 2. Deploy to Google Cloud Dataflow:
bash 01_beam_fundamentals/scripts/02_run_dataflow.sh
```

---

## Summary & What's Next

In this chapter, you learned:
* The **Apache Beam unified execution model** (`Pipeline`, `PCollection`, `PTransform`).
* How to parse raw CSV data and filter rows using `DoFn` and `beam.ParDo`.
* Why **`beam.CombinePerKey`** is the gold standard for high-performance aggregations.
* How to switch seamlessly between local development (`DirectRunner`) and cloud production (`DataflowRunner`).

👉 **Next: [Chapter 02 — Advanced Transformations & BigQuery I/O](../02_transforms_and_bigquery/README.md)**:
We will explore **Side Inputs** for dynamic lookups, **Tagged Multi-Outputs** (dead-letter branches for poison records), and high-throughput streaming/batch writes using the **BigQuery Storage Write API**!